In [ ]:
# macOS note: xgboost and sentence-transformers (PyTorch) each bring their own
# OpenMP runtime. If PyTorch loads first, using xgboost afterwards crashes the
# kernel. Importing xgboost FIRST (before sentence-transformers) avoids the clash.
# Keep this as the very first cell you run.
from xgboost import XGBClassifier

In [ ]:
import pandas as pd

# Load the synthetic patient dataset
df = pd.read_excel('../data/raw/nurseassist_patient_dataset_210_rows.xlsx', sheet_name='Patient Dataset')

# Number of patient rows per illness
df['Primary_Illness'].value_counts()

Primary_Illness
Influenza                          27
Gastrointestinal infection         21
Common cold                        19
Pneumonia                          19
Post-operative wound infection     18
COPD exacerbation                  17
Urinary tract infection            16
Hypertension                       15
Sepsis risk                        15
Strep throat                       14
Skin/wound infection               14
Asthma exacerbation                14
Sepsis risk from skin infection     1
Name: count, dtype: int64

In [ ]:
from sentence_transformers import SentenceTransformer

# Vitals (numeric, model-ready) and the text column we embed
vital_cols = [
    'Temperature_C',
    'Heart_Rate_bpm',
    'Systolic_BP_mmHg',
    'Diastolic_BP_mmHg',
    'SpO2_percent',
    'Respiratory_Rate_bpm',
    'Pain_0_10',
]
label_col = 'Primary_Illness'

# Turn the symptom text into dense embedding vectors
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['Symptoms'].tolist())

# One column per embedding dimension (emb_0, emb_1, ...)
embedding_df = pd.DataFrame(
    embeddings,
    columns=[f'emb_{i}' for i in range(embeddings.shape[1])],
)

# Combine vitals + symptom embeddings + illness label
classification_df = pd.concat(
    [df[vital_cols].reset_index(drop=True), embedding_df, df[label_col].reset_index(drop=True)],
    axis=1,
)

# Replace the processed data file (synthetic / educational only)
classification_df.to_csv('../data/processed/illness_classification_data.csv', index=False)
classification_df.head()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Temperature_C,Heart_Rate_bpm,Systolic_BP_mmHg,Diastolic_BP_mmHg,SpO2_percent,Respiratory_Rate_bpm,Pain_0_10,emb_0,emb_1,emb_2,...,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383,Primary_Illness
0,37.2,82,112,72,98,16,2,0.060930,0.002098,-0.023671,...,0.027558,-0.043411,0.072710,-0.028359,-0.024847,-0.052848,-0.108028,-0.068444,0.049737,Common cold
1,38.6,104,126,78,97,18,5,-0.015175,0.003906,0.019549,...,0.049036,-0.001083,0.027271,-0.007871,-0.016691,-0.061350,-0.059760,-0.067783,0.036219,Strep throat
2,38.9,112,118,74,91,26,6,0.063300,0.065434,0.000812,...,0.051761,-0.018248,0.057586,0.028724,-0.066031,0.037030,-0.045450,-0.053233,0.049096,Pneumonia
3,36.9,88,178,104,96,17,4,-0.030666,-0.067630,0.012032,...,-0.052193,0.080576,0.058733,-0.062322,-0.054040,-0.023859,-0.063322,-0.018578,0.043091,Hypertension
4,38.1,98,132,84,98,18,5,0.027428,-0.023130,0.014981,...,-0.001849,0.012299,0.026669,0.001774,0.006802,0.113743,0.031649,-0.034748,0.009480,Urinary tract infection


In [ ]:
# Some illnesses have too few patients to split and validate.
# Stratified splitting needs >= 2 per class, and StratifiedKFold(5) needs >= 5,
# so we keep only illnesses with at least 5 patients. (synthetic / educational data)
MIN_PER_CLASS = 5

class_counts = classification_df[label_col].value_counts()
keep_classes = class_counts[class_counts >= MIN_PER_CLASS].index

# Keep only illnesses with enough patients
classification_df = classification_df[classification_df[label_col].isin(keep_classes)].reset_index(drop=True)

# Remaining patients per illness after filtering
classification_df[label_col].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

# Features (X) = vitals + symptom embeddings; label (y) = illness
feature_cols = vital_cols + [f'emb_{i}' for i in range(embeddings.shape[1])]
X = classification_df[feature_cols]
y = classification_df[label_col]

# random_state makes the split reproducible (same rows every run)
RANDOM_STATE = 42

# 80-20 split: hold out 20% as the test set we never touch during training.
# stratify=y keeps each illness represented proportionally in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Validation = k-fold cross-validation on the TRAIN set (not a fixed val split).
# StratifiedKFold keeps class balance inside each fold. 5 folds is a common default.
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Confirm the split sizes
pd.DataFrame({
    'rows': [len(X_train), len(X_test)],
}, index=['train', 'test'])

ValueError: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: ['Sepsis risk from skin infection']

In [ ]:
from sklearn.preprocessing import LabelEncoder

# XGBoost needs numeric labels, so turn illness names into integer codes (0, 1, 2, ...).
# This is NOT embedding — the classes carry no similarity, just distinct IDs.
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Map of which illness became which number
dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
import numpy as np

# Baseline XGBoost model. learning_rate=0.1 is a standard starting value.
# (We'll compare other learning rates in the next experiment.)
xgb_model = XGBClassifier(
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    eval_metric='mlogloss',
)

# Validate with 5-fold cross-validation on the training set.
# This gives 5 accuracy scores; the mean is our validation score to compare later.
cv_scores = cross_val_score(xgb_model, X_train, y_train_encoded, cv=kfold, scoring='accuracy')

print(f"Fold accuracies: {np.round(cv_scores, 3)}")
print(f"Mean validation accuracy (learning_rate=0.1): {cv_scores.mean():.3f}")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/srijangupta/Library/Mobile Documents/com~apple~CloudDocs/Personal-Projects/AI-Pharmaceutical-Bioinformatics/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Users/srijangupta/Library/Mobile Documents/com~apple~CloudDocs/Personal-Projects/AI-Pharmaceutical-Bioinformatics/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt

In [ ]:
def evaluate_params(lr, n_estimators=100, max_depth=6):
    """Run 5-fold cross-validation for one XGBoost hyperparameter combo.

    n_estimators and max_depth default to baseline values so we can sweep
    just one setting at a time. Trains on the training set only (test set
    stays untouched). Returns the settings plus the mean validation accuracy
    and its standard deviation across the 5 folds.
    """
    model = XGBClassifier(
        learning_rate=lr,
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=RANDOM_STATE,
        eval_metric='mlogloss',
    )
    scores = cross_val_score(model, X_train, y_train_encoded, cv=kfold, scoring='accuracy')
    return {
        'learning_rate': lr,
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'mean_val_accuracy': scores.mean(),
        'std': scores.std(),
    }

In [ ]:
# --- n_estimators experiment (commented out: proved no effect, see note below) ---
# Grid search: try every combination of learning rate and n_estimators
# using the same 5-fold cross-validation. (Test set stays untouched.)
# results = []
# for lr in [0.3, 0.2, 0.1, 0.05, 0.01]:
#     for n in [100, 300, 500, 1000]:
#         results.append(evaluate_params(lr, n))
#
# # Sorted so the best combination sits at the top
# results_df = pd.DataFrame(results).sort_values('mean_val_accuracy', ascending=False).reset_index(drop=True)
# results_df

In [ ]:
# --- n_estimators plot (commented out: belongs to the n_estimators experiment above) ---
# import matplotlib.pyplot as plt
#
# # One line per n_estimators value, plotted across learning rates.
# fig, ax = plt.subplots(figsize=(8, 5))
#
# for n in sorted(results_df['n_estimators'].unique()):
#     subset = results_df[results_df['n_estimators'] == n].sort_values('learning_rate')
#     ax.plot(
#         subset['learning_rate'],
#         subset['mean_val_accuracy'],
#         marker='o',
#         label=f'n_estimators={n}',
#     )
#
# ax.set_xlabel('Learning rate')
# ax.set_ylabel('Mean validation accuracy (5-fold CV)')
# ax.set_title('XGBoost: learning rate vs validation accuracy by n_estimators')
# ax.legend()
# ax.grid(True, alpha=0.3)
# fig.tight_layout()

### n_estimators experiment — conclusion

The grid search above showed that changing `n_estimators` (100 → 1000) had **no effect** on validation accuracy: for each learning rate, every `n_estimators` value produced the same score. The model converges within 100 trees on this small dataset, so extra trees add nothing.

**Decision:** fix `n_estimators = 100` and do not sweep it further. The cells are kept commented out as a record. Next, we test `max_depth`.

In [ ]:
# max_depth experiment: sweep learning rate x max_depth, with n_estimators fixed at 100.
# Shallow trees often generalize better on small datasets, so we test small depths.
depth_results = []
for lr in [0.3, 0.2, 0.1, 0.05, 0.01]:
    for depth in [2, 3, 4, 6]:
        depth_results.append(evaluate_params(lr, max_depth=depth))

# Sorted so the best combination sits at the top
depth_results_df = pd.DataFrame(depth_results).sort_values('mean_val_accuracy', ascending=False).reset_index(drop=True)
depth_results_df

In [ ]:
import matplotlib.pyplot as plt

# One line per max_depth value, plotted across learning rates,
# so we can see how tree depth and learning rate interact.
fig, ax = plt.subplots(figsize=(8, 5))

for depth in sorted(depth_results_df['max_depth'].unique()):
    subset = depth_results_df[depth_results_df['max_depth'] == depth].sort_values('learning_rate')
    ax.plot(
        subset['learning_rate'],
        subset['mean_val_accuracy'],
        marker='o',
        label=f'max_depth={depth}',
    )

ax.set_xlabel('Learning rate')
ax.set_ylabel('Mean validation accuracy (5-fold CV)')
ax.set_title('XGBoost: learning rate vs validation accuracy by max_depth')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

In [ ]:
# Best combination from the max_depth experiment (top row of the sorted table)
best = depth_results_df.iloc[0]
print(
    f"Best combination: learning_rate={best['learning_rate']}, "
    f"max_depth={int(best['max_depth'])}, n_estimators={int(best['n_estimators'])}"
)
print(f"Mean validation accuracy: {best['mean_val_accuracy']:.3f} (std {best['std']:.3f})")